In [ ]:
import gymnasium as gym
import sys

sys.path.append("../")
from agents.trainers.rainbow_trainer import RainbowTrainer
from configs.games.cartpole import CartPoleConfig
from configs.agents.rainbow_dqn import RainbowConfig
from stats.stats import StatTracker
import torch

# env = ClipReward(AtariPreprocessing(gym.make("MsPacmanNoFrameskip-v4", render_mode="rgb_array"), terminal_on_life_loss=True), -1, 1) # as recommended by the original paper, should already include max pooling
# env = FrameStack(env, 4)
env = gym.make("CartPole-v1")

config_dict = {
    "executor_type": "local",
    "adam_epsilon": 1e-8,
    "learning_rate": 0.001,
    "training_steps": 3000,
    "minibatch_size": 128,
    "transfer_interval": 100,
    "n_step": 3,
    "replay_interval": 4,
    "kernel_initializer": "orthogonal",
    "clipnorm": 10.0,
    "model_name": "rainbow_smoke_test",
    "noisy_sigma": 0.5,
    # --- MISSING ROOT CONFIG PARAMS ---
    "atom_size": 51,
    "v_min": 0.0,
    "v_max": 200.0,  # Curt Park uses 200 for CartPole
    "representation_mode": "categorical",  # CRITICAL: Triggers Bellman Projection
    "weight_decay": 0.0,  # CRITICAL: rainbow_trainer.py crashes without this
    # --- Backbones ---
    "backbone": {
        "type": "mlp",
        "widths": [128],
    },
    # --- Head Configuration (New Nested System) ---
    "head": {
        "output_strategy": {
            "type": "c51",
            "num_atoms": 51,
            # Defaults for v_min/v_max are handled automatically
            # or you can specify them here:
            "v_min": 0,
            "v_max": 500,
        },
        # Hidden layers for Dueling Head (restored to 512 for performance)
        "value_hidden_widths": [],
        "advantage_hidden_widths": [],
        # "noisy_sigma": 0.5 (inherits from global)
    },
    # --- PER (Prioritized Experience Replay) ---
    "per_epsilon": 1e-6,
    "per_alpha": 0.2,
    "per_beta": 0.6,
    "action_selector": {
        "base": {
            "type": "argmax",  # Maps to ArgmaxSelector (Greedy)
            "kwargs": {},
        },
    },
}


game_config = CartPoleConfig()
config = RainbowConfig(config_dict, game_config)
trainer = RainbowTrainer(
    config,
    env,
    torch.device("cpu"),
    "rainbow_refactor",
    StatTracker("rainbow_refactor"),
)

trainer.checkpoint_interval = 100
trainer.test_interval = 500

trainer.train()

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Using         agent_type                    : rainbow
Using         agent_type                    : rainbow
Using default results_path                  : results
Using         training_steps                : 3000
Using         adam_epsilon                  : 1e-08
Using default momentum                      : 0.9
Using         learning_rate                 : 0.001
Using         clipnorm                      : 10.0
Using default optimizer                     : <class 'torch.optim.adam.Adam'>
Using         weight_decay                  : 0.0
Using default num_minibatches               : 1
Using default training_iterations           : 1
Using         lr_schedule                   : None
Using         lr_schedule                   : {'type': 'constant', 'initial': 0.001}
Using default test_interval                 : 1000
Using default checkpoint_interval           : 1000
Using         minibatch_size                : 128
Using default replay_buffer_size            : 5000
Using default min_r

/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/.venv/lib/python3.12/site-packages/gymnasium/core.py:311: UserWarning: WARN: env.possible_agents to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.possible_agents` for environment variables or `env.get_wrapper_attr('possible_agents')` that will search the reminding wrappers.
  logger.warn(


Initializing stat 'score' with subkeys None
Initializing stat 'episode_length' with subkeys None
Initializing stat 'test_score' with subkeys ['avg', 'p0']
Initializing stat 'loss' with subkeys None
Initializing stat 'learner_fps' with subkeys None
Initializing stat 'actor_fps' with subkeys None
Starting Rainbow training for 3000 steps...
Initializing stat 'num_episodes' with subkeys None
Initializing stat 'C51Loss' with subkeys None
Initializing stat 'indices' with subkeys None
Initializing stat 'training_step' with subkeys None
plotting score
plotting episode_length
plotting loss
plotting actor_fps
plotting num_episodes
plotting C51Loss
plotting indices
plotting training_step
Saved stats plot to /Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/test_notebooks/checkpoints/rainbow_refactor/graphs/rainbow_refactor_stats.png
Saved checkpoint at step 100 to /Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/test_notebooks/checkpoints/rainbow_refactor/step_100
Step 100, Epsil

AttributeError: 'NoneType' object has no attribute 'stop'

In [1]:
import sys

sys.path.append("../")
import os
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
import random

from custom_gym_envs.envs.matching_pennies import (
    env as matching_pennies_env,
    MatchingPenniesGymEnv,
)
from agents.trainers.nfsp_trainer import NFSPTrainer
from agent_configs.dqn.nfsp_config import NFSPDQNConfig
from agent_configs.dqn.rainbow_config import RainbowConfig
from game_configs.matching_pennies_config import MatchingPenniesConfig
from learners.losses.basic_losses import (
    KLDivergenceLoss,
    CategoricalCrossentropyLoss,
    HuberLoss,
    MSELoss,
)
import torch
from torch.optim import Adam, SGD
from stats.stats import StatTracker

config_dict = {
    "shared_networks_and_buffers": False,
    "training_steps": 10000,
    "anticipatory_param": 0.1,
    "replay_interval": 2,
    "num_minibatches": 1,
    "learning_rate": 0.1,
    "momentum": 0.0,
    "optimizer": SGD,
    "loss_function": HuberLoss(),
    "min_replay_buffer_size": 500,
    "minibatch_size": 128,
    "replay_buffer_size": 1000,
    "transfer_interval": 300,
    # --- NEW RL BACKBONE ---
    "backbone": {"type": "mlp", "widths": [128]},
    # Note: value_backbone and advantage_backbone default to Identity (None)
    # which replaces the old empty list [] behavior.
    "noisy_sigma": 0.0,
    "eg_epsilon": 0.06,
    "eg_epsilon_decay_type": "inverse_sqrt",
    "eg_epsilon_decay_final_step": 0,
    # --- SL CONFIG ---
    "sl_learning_rate": 0.01,
    "sl_momentum": 0.0,
    "sl_optimizer": SGD,
    "sl_loss_function": CategoricalCrossentropyLoss(),
    "sl_min_replay_buffer_size": 500,
    "sl_minibatch_size": 128,
    "sl_replay_buffer_size": 20000,
    # --- NEW SL BACKBONE ---
    "sl_backbone": {"type": "mlp", "widths": [128]},
    "sl_clip_low_prob": 0.0,
    "per_alpha": 0.0,
    "per_beta": 0.0,
    "per_beta_final": 0.0,
    "per_epsilon": 0.00001,
    "n_step": 1,
    "atom_size": 1,
    "dueling": False,
    "clipnorm": 10.0,
    "sl_clipnorm": 10.0,
}

config = NFSPDQNConfig(
    config_dict=config_dict,
    game_config=MatchingPenniesConfig(),
)
config.save_intermediate_weights = True

import custom_gym_envs
import gymnasium as gym

# env = gym.make("custom_gym_envs/LeducHoldem-v0", encode_player_turn=False)

env = matching_pennies_env(render_mode="rgb_array", max_cycles=1)

trainer = NFSPTrainer(config, env, torch.device("cpu"), StatTracker("nfsp_3"))
trainer.checkpoint_interval = 100
trainer.test_interval = 1000
trainer.test_trials = 10000
trainer.train()

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


NameError: name 'Optional' is not defined